In [ ]:
from google.colab import drive
# A prior Colab FUSE session can be stale after an idle disconnect.
drive.mount('/content/drive', force_remount=True)

# EXP-110P — semantic label-prototype/cache specialist

CPU is primary. This notebook consumes only the compact, hash-verified bundle, uses strict inner F1–F4 contexts, and stops before Fold 0. It is not lexical query-memory and does not use EXP-109C late-interaction scores.

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/DSC2026/LegalIR/exp110p')
RUN_ID = 'exp110p-' + __import__('datetime').datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
USE_GPU_FOR_SIMILARITY = False
OUTER = 'fold_0'
assert OUTER == 'fold_0'
print('RUN_ID=', RUN_ID, 'DRIVE_ROOT=', DRIVE_ROOT, 'USE_GPU_FOR_SIMILARITY=', USE_GPU_FOR_SIMILARITY, flush=True)

In [ ]:
# Bootstrap: all required directories are created by the notebook.
DIRS = [
    DRIVE_ROOT / 'input', DRIVE_ROOT / 'code', DRIVE_ROOT / 'cache',
    DRIVE_ROOT / 'cache/similarity', DRIVE_ROOT / 'cache/prototypes',
    DRIVE_ROOT / 'cache/features', DRIVE_ROOT / 'cache/predictions',
    DRIVE_ROOT / 'results', DRIVE_ROOT / f'results/logs/{RUN_ID}',
    DRIVE_ROOT / 'checkpoints', DRIVE_ROOT / 'exports', DRIVE_ROOT / 'invalidated',
]
for directory in DIRS:
    directory.mkdir(parents=True, exist_ok=True)
print('phase=bootstrap state=PASS directories=', len(DIRS), flush=True)

In [ ]:
import datetime, json, os, sys, time, shutil

class DriveLogger:
    def __init__(self, root, run_id):
        self.root, self.run_id = Path(root), run_id
        self.run_dir = self.root / 'results' / 'logs' / run_id
        self.run_dir.mkdir(parents=True, exist_ok=True)
    def log(self, phase, message):
        line = f'[{datetime.datetime.now(datetime.UTC).isoformat()}] phase={phase} {message}'
        for path in (self.run_dir / 'run.log', self.run_dir / f'{phase}.log'):
            with path.open('a', encoding='utf-8', newline='\n', buffering=1) as handle:
                handle.write(line + '\n'); handle.flush()
        print(line, flush=True)

LOGGER = DriveLogger(DRIVE_ROOT, RUN_ID)
def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name('.' + path.name + '.tmp')
    with tmp.open('w', encoding='utf-8', newline='\n') as handle:
        json.dump(value, handle, ensure_ascii=False, sort_keys=True, indent=2); handle.write('\n'); handle.flush(); os.fsync(handle.fileno())
    tmp.replace(path)

def phase_receipt(phase, payload, completed, total, *, run_state='PASS'):
    result_dir = DRIVE_ROOT / 'results' / phase
    result_dir.mkdir(parents=True, exist_ok=True)
    payload = dict(payload); payload.update({'schema_version': 'legalir.exp110p_semantic_label_prototype.v1', 'phase': phase, 'run_id': RUN_ID, 'completed': int(completed), 'total': int(total), 'finished_at': datetime.datetime.now(datetime.UTC).isoformat()})
    atomic_json(result_dir / 'phase_manifest.json', payload)
    import hashlib
    digest = hashlib.sha256((result_dir / 'phase_manifest.json').read_bytes()).hexdigest()
    atomic_json(result_dir / '_SUCCESS.json', {'status': 'PASS', 'phase': phase, 'sha256': digest, 'run_id': RUN_ID})
    atomic_json(DRIVE_ROOT / 'results' / 'RUN_STATUS.json', {'run_id': RUN_ID, 'state': str(run_state), 'phase': phase, 'completed': int(completed), 'total': int(total), 'fold0_read': False, 'last_heartbeat': datetime.datetime.now(datetime.UTC).isoformat()})
    LOGGER.log(phase, f'completed={completed}/{total} state={run_state} result_path={result_dir}')


In [ ]:
# Do not replace NumPy/SciPy binaries inside a live Colab kernel: that can break
# LightGBM's compiled extension. Record the compatible runtime and make the
# later exact anchor-ranking gate, rather than a version string, authoritative.
import hashlib, importlib.metadata
RUNTIME_PACKAGES = ('numpy', 'scipy', 'scikit-learn', 'lightgbm', 'psutil', 'torch')
runtime_versions = {package: importlib.metadata.version(package) for package in RUNTIME_PACKAGES}
try:
    import numpy, scipy, sklearn, lightgbm
except Exception as exc:
    raise RuntimeError('REJECTED_DEPENDENCY_GATE: restart the Colab runtime, then use its compatible preinstalled NumPy/SciPy/LightGBM set; do not pip-upgrade scientific packages in-place') from exc
LOGGER.log('bootstrap', 'runtime_imports_verified versions=' + json.dumps(runtime_versions, sort_keys=True))
# Verify the core with stdlib before importing it from Drive.
bootstrap_manifest = json.loads((DRIVE_ROOT / 'input/INPUT_MANIFEST.json').read_text(encoding='utf-8'))
for code_name, code_entry in bootstrap_manifest.get('code_files', {}).items():
    code_path = DRIVE_ROOT / 'code' / code_name
    actual = hashlib.sha256(code_path.read_bytes()).hexdigest() if code_path.exists() else None
    if actual != str(code_entry.get('sha256')) or code_path.stat().st_size != int(code_entry.get('bytes', -1)):
        raise RuntimeError(f'REJECTED_INPUT_OR_LEAKAGE_GATE unverified core: {code_name}')
sys.path.insert(0, str(DRIVE_ROOT / 'code'))
import exp110p_semantic_label_prototype as core
LOGGER.log('bootstrap', 'core_hash_verified_and_loaded path=' + str(DRIVE_ROOT / 'code/exp110p_semantic_label_prototype.py'))

## Phase 0 — input hash and leakage verification
Verify the compact bundle before opening experiment data.

In [ ]:
# Phase 0/1: verify every input hash before opening experiment data.
LOGGER.log('input_hash_verify', 'START completed=0/0 ETA=unknown RAM_available=measuring gate=PENDING')
REQUIRED = [
 'train.json', 'cv_folds.json', 'exclusions.json', 'label_impact_report.json',
 'e5_query_embeddings/train_queries.f32.npy', 'e5_query_embeddings/train_query_ids.json', 'e5_query_embeddings/manifest.json',
 'exp109b_sources_top50.jsonl', 'exp109b_anchor_inner_predictions.jsonl', 'exp109b_locked_configs.json', 'exp109b_pilot_report.json',
 'parent_metadata_minimal.jsonl', 'exp024_report.json'
]
manifest = core.read_json(DRIVE_ROOT / 'input/INPUT_MANIFEST.json')
def verify_with_drive_recovery():
    # Never accept an unhashed bundle: repair a transient Drive FUSE loss once, then fail closed.
    for attempt in (1, 2):
        try:
            return core.verify_input_manifest(DRIVE_ROOT / 'input', manifest, REQUIRED)
        except OSError as exc:
            if attempt == 2 or getattr(exc, 'errno', None) != 107:
                raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE: Drive is unavailable; remount/re-run Phase 0') from exc
            LOGGER.log('input_hash_verify', 'Drive FUSE disconnected; force-remounting once before retry')
            drive.mount('/content/drive', force_remount=True)
    raise AssertionError('unreachable')
hash_report = verify_with_drive_recovery()
if manifest.get('label_fingerprint') != core.LABEL_FINGERPRINT or manifest.get('fold0_predictions_included') is not False or int(manifest.get('source_rows_feature_context_top_k', 0)) < 500:
    raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE')
embedding_manifest = core.read_json(DRIVE_ROOT / 'input/e5_query_embeddings/manifest.json')
if embedding_manifest.get('dimension') != core.DIMENSION or embedding_manifest.get('dtype') != 'float32' or embedding_manifest.get('model_id') != 'mainguyen9/vietlegal-e5':
    raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE embedding manifest contract')
for artifact_name, expected_hash in embedding_manifest.get('artifacts', {}).items():
    artifact_path = DRIVE_ROOT / 'input/e5_query_embeddings' / artifact_name
    if not artifact_path.exists() or core.sha256_file(artifact_path) != str(expected_hash):
        raise RuntimeError(f'REJECTED_INPUT_OR_LEAKAGE_GATE embedding artifact hash: {artifact_name}')
for code_name, code_entry in manifest.get('code_files', {}).items():
    code_path = DRIVE_ROOT / 'code' / code_name
    if not code_path.exists() or code_path.stat().st_size != int(code_entry['bytes']) or core.sha256_file(code_path) != str(code_entry['sha256']):
        raise RuntimeError(f'REJECTED_INPUT_OR_LEAKAGE_GATE code hash: {code_name}')
LOGGER.log('input_hash_verify', f"completed={len(hash_report['files'])}/{len(REQUIRED)} throughput=verified_files/s cache={DRIVE_ROOT / 'input'} gate=PASS")
phase_receipt('input_hash_verify', {'manifest_sha256': core.sha256_file(DRIVE_ROOT / 'input/INPUT_MANIFEST.json'), 'fold0_read': False}, len(REQUIRED), len(REQUIRED))

## Phase 1 — Colab resource preflight
CPU is primary; optional GPU similarity is measured and budget-gated.

In [ ]:
# Phase 1: resource preflight. GPU is opt-in and is disabled if its measured projection exceeds one hour.
LOGGER.log('preflight', 'START completed=0/1 ETA=measuring RAM_available=measuring gate=PENDING')
import numpy as np, psutil
snapshot = core.resource_snapshot()
drive_free = shutil.disk_usage(DRIVE_ROOT).free
vectors = np.load(DRIVE_ROOT / 'input/e5_query_embeddings/train_queries.f32.npy', mmap_mode='r', allow_pickle=False)
preflight_query_ids = [str(x) for x in core.read_json(DRIVE_ROOT / 'input/e5_query_embeddings/train_query_ids.json')]
preflight_folds, _preflight_fold_for = core.load_folds(DRIVE_ROOT / 'input/cv_folds.json')
preflight_inner_ids = core.inner_qids(preflight_folds, outer=OUTER)
preflight_qrow = {qid: i for i, qid in enumerate(preflight_query_ids)}
preflight_inner_rows = [preflight_qrow[qid] for qid in preflight_inner_ids]
bench = np.asarray(vectors[preflight_inner_rows[:256]], dtype=np.float32)
started = time.perf_counter(); _ = core.cosine_matrix(bench, bench); elapsed = max(time.perf_counter() - started, 1e-9)
projected_cpu_seconds = elapsed * (5600 / 256) * 1.5
gpu_projected_seconds = None
projected_inner_query_count = len(preflight_inner_rows)
gpu_budget = {'requested': bool(USE_GPU_FOR_SIMILARITY), 'available': bool(snapshot.get('gpu_available', False)), 'used': False, 'status': 'CPU_PRIMARY'}
if USE_GPU_FOR_SIMILARITY and snapshot.get('gpu_available', False):
    gpu_started = time.perf_counter(); _ = core.gpu_similarity_block(bench[:min(20, len(bench))], np.asarray(vectors[preflight_inner_rows], dtype=np.float32)); gpu_elapsed = max(time.perf_counter() - gpu_started, 1e-9)
    gpu_projected_seconds = gpu_elapsed * (projected_inner_query_count / max(min(20, len(bench)), 1)) * 1.2
    if gpu_projected_seconds is not None and gpu_projected_seconds > 3600:
        USE_GPU_FOR_SIMILARITY = False; gpu_budget.update({'status': 'STOP_GPU_BUDGET', 'projected_seconds': gpu_projected_seconds}); LOGGER.log('preflight', f'STOP_GPU_BUDGET projected_gpu_seconds={gpu_projected_seconds:.2f}; CPU primary')
    else:
        gpu_budget.update({'status': 'GPU_WITHIN_ONE_HOUR', 'used': True, 'benchmark_seconds': gpu_elapsed, 'projected_seconds': gpu_projected_seconds})
projected_peak = int(5600 * 5600 * 4 + 7000 * 1024 * 4 + 350 * 1024**2)
resource_gate = core.projected_resource_gate(snapshot, projected_peak_bytes=projected_peak, projected_cpu_seconds=projected_cpu_seconds, drive_free_bytes=drive_free)
LOGGER.log('preflight', f"completed=1/1 throughput={1/max(elapsed,1e-9):.2f} bench/s ETA_seconds={projected_cpu_seconds:.2f} RAM_available={snapshot.get('ram_available_bytes')} GPU={snapshot.get('gpu_available', False)} gate={resource_gate['status']}")
if USE_GPU_FOR_SIMILARITY and not snapshot.get('gpu_available', False):
    USE_GPU_FOR_SIMILARITY = False; gpu_budget.update({'status': 'STOP_GPU_BUDGET', 'reason': 'gpu_unavailable'}); LOGGER.log('preflight', 'STOP_GPU_BUDGET gpu_unavailable; CPU primary')
import importlib.metadata
def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return 'MISSING'
versions = {'numpy': np.__version__, 'scipy': __import__('scipy').__version__, 'scikit-learn': installed_version('scikit-learn'), 'lightgbm': installed_version('lightgbm'), 'psutil': installed_version('psutil'), 'torch': installed_version('torch')}
atomic_json(DRIVE_ROOT / 'results/PREFLIGHT.json', {'versions': versions, 'snapshot': snapshot, 'drive_free_bytes': drive_free, 'block_benchmark_seconds': elapsed, 'projected_cpu_seconds': projected_cpu_seconds, 'projected_peak_bytes': projected_peak, 'gate': resource_gate, 'gpu_budget': gpu_budget, 'gpu_budget_rule_seconds': 3600, 'fold0_read': False})
if resource_gate['status'] != 'PASS_COLAB_RESOURCE_GATE':
    raise RuntimeError('REJECTED_COLAB_RESOURCE_GATE')
phase_receipt('preflight', {'gate': resource_gate, 'projected_cpu_seconds': projected_cpu_seconds, 'gpu_budget': gpu_budget, 'use_gpu_for_similarity': USE_GPU_FOR_SIMILARITY}, 1, 1)

## Phase 2 — canonical labels and leakage audit
Recompute labels and folds from the verified inputs.

In [ ]:
# Phase 2: recompute canonical labels and fold coverage; manifest values are not trusted blindly.
LOGGER.log('canonical_audit', 'START completed=0/4 ETA=measuring RAM_available=measuring gate=PENDING')
input_dir = DRIVE_ROOT / 'input'
answers, label_stats = core.canonical_labels(input_dir / 'train.json', input_dir / 'exclusions.json', input_dir / 'label_impact_report.json')
train_rows = core.read_json(input_dir / 'train.json')
query_token_lengths = {str(qid): len(str(row.get('question', '')).split()) for qid, row in train_rows.items()}
folds, fold_for = core.load_folds(input_dir / 'cv_folds.json')
core.validate_fold_partition(folds, answers, outer=OUTER)
if (label_stats['queries'], label_stats['evaluable_queries'], label_stats['non_evaluable_queries'], label_stats['assignment_count'], label_stats['label_fingerprint']) != (7000, 6991, 9, 7626, core.LABEL_FINGERPRINT):
    raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE canonical counts/fingerprint')
inner_ids = core.inner_qids(folds, outer=OUTER); outer_ids = set(folds[OUTER])
query_ids = [str(x) for x in core.read_json(input_dir / 'e5_query_embeddings/train_query_ids.json')]
if set(query_ids) != set(answers) or len(query_ids) != 7000:
    raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE embedding IDs')
if vectors.shape != (7000, 1024) or not np.isfinite(np.asarray(vectors[[query_ids.index(qid) for qid in inner_ids[:32]]])).all():
    raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE embedding shape/finite')
if outer_ids & set(inner_ids):
    raise RuntimeError('Fold 0 entered inner qids')
LOGGER.log('canonical_audit', f"completed=4/4 throughput=labels/s ETA=0 RAM_available={psutil.virtual_memory().available} gate=PASS")
atomic_json(DRIVE_ROOT / 'results/CANONICAL_AUDIT.json', {'label_stats': label_stats, 'folds': {k: len(v) for k, v in folds.items()}, 'embedding_shape': list(vectors.shape), 'inner_query_count': len(inner_ids), 'fold0_read': False, 'status': 'PASS'})
phase_receipt('canonical_audit', {'label_stats': label_stats, 'inner_query_count': len(inner_ids), 'fold0_read': False}, 4, 4)

## Phase 3 — EXP-109B anchor reproduction
Reproduce the F1–F4 anchor exactly before prototype construction.

In [ ]:
# Phase 3: source/anchor load and exact EXP-109B anchor contract.
LOGGER.log('anchor_reproduction', 'START completed=0/5593 ETA=measuring RAM_available=measuring gate=PENDING')
source_rows = {str(row['qid']): row for row in core.read_jsonl(input_dir / 'exp109b_sources_top50.jsonl')}
if set(source_rows) != set(inner_ids):
    raise RuntimeError('REJECTED_ANCHOR_REPRODUCTION_GATE source rows must be inner-only F1-F4')
parent_meta = {str(row['doc_id']): row for row in core.read_jsonl(input_dir / 'parent_metadata_minimal.jsonl')}
known_docs = set(parent_meta)
for qid, row in source_rows.items():
    for source in core.SOURCES:
        required_context = 500 if source != 'bm25' else 50
        if len(row.get('sources', {}).get(source, [])) < required_context:
            raise RuntimeError(f'REJECTED_ANCHOR_REPRODUCTION_GATE feature context {qid}/{source}')
        if not {str(x['doc_id']) for x in row['sources'][source]} <= known_docs:
            raise RuntimeError('candidate doc outside parent universe')
anchor_predictions, anchor_raw = core.load_anchor_prediction_rows(input_dir / 'exp109b_anchor_inner_predictions.jsonl')
core.validate_inner_only_qids(anchor_predictions, folds, outer=OUTER)
if set(anchor_predictions) != set(inner_ids):
    raise RuntimeError('REJECTED_ANCHOR_REPRODUCTION_GATE anchor must cover exactly F1-F4')
configs = core.read_json(input_dir / 'exp109b_locked_configs.json')
if configs.get('fold0_included') is not False or set(configs.get('configs_by_heldout_fold', {})) != set(core.INNER_FOLDS):
    raise RuntimeError('REJECTED_ANCHOR_REPRODUCTION_GATE locked configs')
anchor_cache_key = core.content_hash({'input_manifest_sha256': core.sha256_file(input_dir / 'INPUT_MANIFEST.json'), 'core_sha256': manifest['code_files']['exp110p_semantic_label_prototype.py']['sha256'], 'anchor_sha256': manifest['files']['exp109b_anchor_inner_predictions.jsonl']['sha256'], 'source_rows_sha256': manifest['files']['exp109b_sources_top50.jsonl']['sha256'], 'locked_configs_sha256': manifest['files']['exp109b_locked_configs.json']['sha256'], 'contract': 'exact-ranking-identity-v1', 'fold0_read': False})
anchor_cache_path = DRIVE_ROOT / 'results/ANCHOR_REPRODUCTION.json'
cached_anchor = core.read_json(anchor_cache_path) if anchor_cache_path.exists() else {}
if cached_anchor.get('status') == 'PASS' and cached_anchor.get('cache_key') == anchor_cache_key and cached_anchor.get('fold0_read') is False and cached_anchor.get('raw_score_contract', {}).get('ranking_identity') is True:
    reproduced_raw, reproduction_meta = {}, {'cache_hit': True, 'cache_key': anchor_cache_key, 'fold0_read': False}
    LOGGER.log('anchor_reproduction', 'CACHE_HIT exact verified ranking identity; LambdaMART replay skipped')
else:
    reproduced_anchor, reproduced_raw, reproduction_meta = core.reproduce_anchor_predictions(source_rows, answers, folds, configs['configs_by_heldout_fold'], outer=OUTER, depth=50, parent_metadata=parent_meta, query_token_lengths=query_token_lengths)
    if any(reproduced_anchor[qid] != anchor_predictions[qid] for qid in inner_ids):
        raise RuntimeError('REJECTED_ANCHOR_REPRODUCTION_GATE prediction order/set is not identical to exported anchor')
anchor_metrics = core.prediction_metrics(anchor_predictions, answers, inner_ids)
expected_anchor = {'recall@5': 0.9251057869956494, 'recall@1': 0.6753531199713928, 'mrr@5': 0.8044072948328268, 'precision@5': 0.19749687108886105, 'multi_gold_recall@5': 0.7174585218702866}
if any(abs(float(anchor_metrics[k]) - value) > 1e-12 for k, value in expected_anchor.items()):
    raise RuntimeError(f'REJECTED_ANCHOR_REPRODUCTION_GATE metrics={anchor_metrics}')
LOGGER.log('anchor_reproduction', f"completed=5593/5593 throughput=predictions/s ETA=0 RAM_available={psutil.virtual_memory().available} gate=PASS")
if not reproduction_meta.get('cache_hit'):
    atomic_json(anchor_cache_path, {'cache_key': anchor_cache_key, 'metrics': anchor_metrics, 'expected': expected_anchor, 'configs': configs, 'reproduction_meta': reproduction_meta, 'raw_score_contract': {'exported_raw_scores_present': any(bool(anchor_raw.get(qid)) for qid in inner_ids), 'reproduced_raw_scores_present': any(bool(reproduced_raw.get(qid)) for qid in inner_ids), 'numeric_equality_claim': False, 'ranking_identity': True}, 'candidate_contract': 'union_top50_per_source', 'fold0_read': False, 'status': 'PASS'})
phase_receipt('anchor_reproduction', {'metrics': anchor_metrics, 'candidate_contract': 'union_top50_per_source', 'fold0_read': False}, len(inner_ids), len(inner_ids))

## Phase 4 — inner-only FP32 similarity cache
Write verified, resumable F1–F4 shards; Fold 0 is excluded.

In [ ]:
# Phase 4: inner-only FP32 similarity shards. Fold 0 vectors are not written to this cache.
LOGGER.log('similarity_cache', 'START completed=0/unknown ETA=measuring RAM_available=measuring gate=PENDING')
inner_rows = [query_ids.index(qid) for qid in inner_ids]
inner_vectors = np.asarray(vectors[inner_rows], dtype=np.float32)
similarity_manifest = core.write_similarity_shards(inner_vectors, inner_ids, DRIVE_ROOT / 'cache/similarity', block_size=256, forbidden_qids=outer_ids, resume=True)
if set(similarity_manifest['qids']) & outer_ids or similarity_manifest['fold0_read'] is not False:
    raise RuntimeError('REJECTED_INPUT_OR_LEAKAGE_GATE similarity Fold 0 contamination')
gpu_parity = []
if USE_GPU_FOR_SIMILARITY:
    if len(similarity_manifest['shards']) < 20:
        raise RuntimeError('REJECTED_COLAB_RESOURCE_GATE GPU parity requires at least 20 blocks')
    parity_rows = 20
    for entry in similarity_manifest['shards'][:parity_rows]:
        start, stop = int(entry['start']), int(entry['stop'])
        cpu_block = np.asarray(inner_vectors[start:stop] @ core.normalize_fp32(inner_vectors).T, dtype=np.float32)
        gpu_block = core.gpu_similarity_block(inner_vectors[start:stop], inner_vectors)
        parity = core.similarity_parity(cpu_block, gpu_block, inner_ids)
        gpu_parity.append(parity)
        if not parity['pass']:
            USE_GPU_FOR_SIMILARITY = False; LOGGER.log('similarity_cache', 'GPU parity failed; CPU FP32 retained'); break
LOGGER.log('similarity_cache', f"completed={len(similarity_manifest['shards'])}/{len(similarity_manifest['shards'])} throughput=shards/s ETA=measured CPU={not USE_GPU_FOR_SIMILARITY} GPU={USE_GPU_FOR_SIMILARITY} cache={DRIVE_ROOT / 'cache/similarity'} gate=PASS")
phase_receipt('similarity_cache', {'manifest': similarity_manifest, 'gpu_parity': gpu_parity, 'fold0_read': False, 'gpu_used': USE_GPU_FOR_SIMILARITY}, len(similarity_manifest['shards']), len(similarity_manifest['shards']))

## Phases 5–7 — fold-safe prototype features and source screen
Select one Tip-Adapter policy per heldout inner fold and record support provenance.

In [ ]:
# Phase 5–7: strict nested prototype source screen. The 27-policy grid is selected inside each heldout inner fold.
LOGGER.log('prototype_source_screen', 'START completed=0/4 ETA=measuring RAM_available=measuring gate=PENDING')
qrow = {qid: i for i, qid in enumerate(query_ids)}
# Keep only aggregate coverage: retaining all per-document feature dicts is OOM-prone.
prototype_coverage = {'queries': 0, 'candidate_documents': 0, 'seen_candidate_documents': 0}
support_sidecars, bank_cache = {}, {}
def get_bank(support_qids):
    key = tuple(sorted(str(qid) for qid in support_qids))
    if key not in bank_cache:
        bank_cache[key] = core.build_support_bank(key, vectors, qrow, answers, forbidden_qids=outer_ids, support_fold_names=sorted({fold_for[qid] for qid in key}))
    bank = bank_cache[key]
    support_sidecars.update(bank.sidecar())
    return bank
def proto_predictions(target_qids, support_qids, policy, *, expansion=False):
    bank = get_bank(support_qids)
    support_sidecars.update(bank.sidecar())
    result, expansions = {}, {}
    for qid in sorted(target_qids):
        base = core.candidate_union(source_rows[qid], depth=50)
        # Grid selection needs only frozen base candidates.  Full prototype-doc expansion is computed once after policy lock.
        all_docs = sorted(set(base) | set(bank.support_by_doc)) if expansion else base
        features = core.prototype_feature_records(vectors[qrow[qid]], all_docs, bank, soft_policy=policy, source_rows=source_rows[qid])
        prototype_coverage['queries'] += 1
        prototype_coverage['candidate_documents'] += len(features)
        prototype_coverage['seen_candidate_documents'] += sum(float(record.get('proto_seen', 0.0)) > 0.5 for record in features.values())
        views = core.prototype_views(features)
        ranked_views = {name: core.rank_scores(scores) for name, scores in views.items()}
        # Metrics use at most @50; persist no full support-document ranking.
        result[qid] = {name: ranked[:50] for name, ranked in ranked_views.items()}
        base_set = set(base)
        expansions[qid] = {name: [doc for doc in ranked if doc not in base_set][:10] for name, ranked in ranked_views.items()}
        del features, views, ranked_views
    return result, expansions, bank

screen_rows, selected_rows, expansion_rows = [], [], []
for heldout in core.INNER_FOLDS:
    train_folds = [fold for fold in core.INNER_FOLDS if fold != heldout]
    policy_scores = {}
    for policy in core.SOFT_POLICY_GRID:
        values = []
        for validation in train_folds:
            support = [qid for fold in train_folds if fold != validation for qid in folds[fold]]
            pred, _exp, _bank = proto_predictions(folds[validation], support, policy, expansion=False)
            values.append(core.prediction_metrics({qid: pred[qid]['P4_fixed_normalized_combination'] for qid in pred}, answers, folds[validation])['recall@5'])
        policy_scores[policy] = float(np.mean(values))
    chosen_policy = max(policy_scores, key=lambda policy: (policy_scores[policy], -core.SOFT_POLICY_GRID.index(policy)))
    support = [qid for fold in train_folds for qid in folds[fold]]
    pred, expansions, bank = proto_predictions(folds[heldout], support, chosen_policy, expansion=True)
    for view in ('P1_max_exemplar', 'P2_normalized_centroid', 'P3_soft_cache_vote', 'P4_fixed_normalized_combination'):
        metrics = core.prediction_metrics({qid: pred[qid][view] for qid in pred}, answers, folds[heldout])
        screen_rows.append({'heldout': heldout, 'view': view, 'policy': list(chosen_policy), 'metrics': metrics, 'support': bank.provenance(), 'fold0_read': False})
    expansion_rows.append({'heldout': heldout, 'policy': list(chosen_policy), 'novel_top10': expansions, 'base_candidate_count': {qid: len(core.candidate_union(source_rows[qid], depth=50)) for qid in folds[heldout]}, 'expanded_candidate_count': {qid: len(set(core.candidate_union(source_rows[qid], depth=50)) | set(expansions[qid]['P4_fixed_normalized_combination'])) for qid in folds[heldout]}, 'fold0_read': False})
    selected_rows.append({'heldout': heldout, 'policy': list(chosen_policy), 'support_fingerprint': bank.support_fingerprint, 'support_query_count': len(bank.support_qids), 'fold0_read': False})

novel_gold_occurrences_by_fold = {}
novel_gold_queries_by_fold = {}
for row in expansion_rows:
    heldout = row['heldout']; novel = row['novel_top10']['P4_fixed_normalized_combination']; occurrences = [doc for qid, docs in novel.items() for doc in docs if doc in answers.get(qid, set())]
    novel_gold_occurrences_by_fold[heldout] = len(occurrences)
    novel_gold_queries_by_fold[heldout] = sum(any(doc in answers.get(qid, set()) for doc in docs) for qid, docs in novel.items())
expansion_gate = {'eligible': sum(novel_gold_occurrences_by_fold.values()) >= 20 and all(value >= 2 for value in novel_gold_occurrences_by_fold.values()), 'total_novel_gold_occurrences': sum(novel_gold_occurrences_by_fold.values()), 'novel_gold_occurrences_by_fold': novel_gold_occurrences_by_fold, 'novel_gold_queries_by_fold': novel_gold_queries_by_fold, 'rule': 'total>=20 and every inner fold>=2', 'fold0_read': False}
atomic_json(DRIVE_ROOT / 'cache/prototypes/support_sidecar.json', support_sidecars)
prototype_coverage['missing_candidate_documents'] = prototype_coverage['candidate_documents'] - prototype_coverage['seen_candidate_documents']
prototype_coverage['seen_candidate_fraction'] = float(prototype_coverage['seen_candidate_documents'] / prototype_coverage['candidate_documents']) if prototype_coverage['candidate_documents'] else 0.0
prototype_report = {'schema_version': core.SCHEMA, 'stage': 'prototype_source_screen', 'selection_scope': 'nested F1-F4 only', 'views': screen_rows, 'selected_policies': selected_rows, 'candidate_expansion': expansion_rows, 'expansion_gate': expansion_gate, 'prototype_coverage': prototype_coverage, 'support_sidecar_sha256': core.sha256_file(DRIVE_ROOT / 'cache/prototypes/support_sidecar.json'), 'exp024_comparison': core.read_json(input_dir / 'exp024_report.json'), 'fold0_read': False, 'status': 'PASS'}
atomic_json(DRIVE_ROOT / 'results/PROTOTYPE_SOURCE_SCREEN.json', prototype_report)
LOGGER.log('prototype_source_screen', f"completed=4/4 throughput=folds/s ETA=measured RAM_available={psutil.virtual_memory().available} cache={DRIVE_ROOT / 'cache/prototypes'} gate=PASS")
phase_receipt('prototype_source_screen', {'selected_policies': selected_rows, 'expansion_gate': expansion_gate, 'fold0_read': False}, 4, 4)

## Phases 8–9 — nested arms and strict inner gate
Lock arm/configuration in training context, score F1–F4 once, then stop.

In [ ]:
# Phase 8–9: strict nested arm choice. For each heldout H, arm selection uses only the three
LOGGER.log('strict_inner_gate', 'START completed=0/5593 ETA=measuring RAM_available=measuring gate=PENDING')
# training folds; H is scored once after the arm is locked. B uses leave-one-query-out prototype
# rows, C uses the protected residual, and D adds at most ten prototype-only documents.
anchor_inner = {qid: anchor_predictions[qid] for qid in inner_ids}
anchor_rank_score = lambda qid: anchor_raw.get(qid) or {doc: -float(rank) for rank, doc in enumerate(anchor_inner[qid], 1)}

def make_residual_predictions(target_qids, support_qids, policy, *, alpha=0.10, threshold_value=None, anchor_high_confidence_threshold=None):
    result = {}
    bank = get_bank(support_qids)
    for qid in sorted(target_qids):
        docs = core.candidate_union(source_rows[qid], depth=50)
        scores_anchor = anchor_rank_score(qid)
        features = core.prototype_feature_records(vectors[qrow[qid]], docs, bank, soft_policy=policy, source_rows=source_rows[qid], anchor_scores=scores_anchor)
        sample = next(iter(features.values()))
        threshold = float(threshold_value) if threshold_value is not None else (float(np.quantile([features[doc]['nearest_support_similarity'] for doc in docs], .70)) if docs else core.MISSING_SENTINEL)
        anchor_ordered = sorted(scores_anchor.values(), reverse=True)
        anchor_confidence = float(anchor_ordered[0] - anchor_ordered[1]) if len(anchor_ordered) > 1 else 0.0
        weight = core.confidence_gate_weight(nearest_similarity=sample['nearest_support_similarity'], prototype_winner_margin=sample['prototype_winner_margin'], prototype_vote_entropy=sample['prototype_vote_entropy'], seen_candidate_fraction=sample['prototype_seen_candidate_fraction'], threshold=threshold, anchor_confidence=anchor_confidence, anchor_high_confidence_threshold=anchor_high_confidence_threshold)
        proto_scores = {doc: -float(rank) for rank, doc in enumerate(core.rank_scores({doc: features[doc]['proto_top2_mean'] for doc in docs}), 1)}
        final_scores = core.protected_residual_scores(scores_anchor, proto_scores, alpha=alpha, gate_weights={doc: weight for doc in set(scores_anchor) | set(proto_scores)})
        result[qid] = core.rank_scores(final_scores)
    return result

def residual_context_thresholds(train_folds, policy):
    nearest_values, anchor_confidences = [], []
    for validation in train_folds:
        support = [qid for fold in train_folds if fold != validation for qid in folds[fold]]
        bank = get_bank(support)
        for qid in folds[validation]:
            query = core.normalize_fp32(np.asarray(vectors[qrow[qid]], dtype=np.float32).reshape(1, -1))[0]
            nearest_values.append(float(np.max(bank.vectors @ query)) if len(bank.support_qids) else core.MISSING_SENTINEL)
            ordered = sorted(anchor_rank_score(qid).values(), reverse=True)
            anchor_confidences.append(float(ordered[0] - ordered[1]) if len(ordered) > 1 else 0.0)
    return {quantile: float(np.quantile(nearest_values, quantile)) for quantile in (.50, .70, .85)}, float(np.quantile(anchor_confidences, .85))

def make_expansion_predictions(target_qids, support_qids, policy, view='P4_fixed_normalized_combination'):
    pred, expansions, _bank = proto_predictions(target_qids, support_qids, policy)
    result = {}
    for qid in sorted(target_qids):
        base = set(core.candidate_union(source_rows[qid], depth=50))
        allowed = base | set(expansions[qid][view][:10])
        result[qid] = [doc for doc in pred[qid][view] if doc in allowed]
    return result

def evaluate_b_context(heldout, policy):
    train_folds = [fold for fold in core.INNER_FOLDS if fold != heldout]
    b_configs = core.bounded_lambdamart_configs(configs['configs_by_heldout_fold'][heldout])
    predictions_by_config = {index: {} for index in range(len(b_configs))}
    for validation in train_folds:
        support = [qid for fold in train_folds if fold != validation for qid in folds[fold]]
        valid = [str(qid) for qid in folds[validation]]
        X, y, groups, _ids = core.make_augmented_training_rows(support, source_rows, answers, vectors, qrow, support_qids_for_qid={qid: [x for x in support if x != qid] for qid in support}, policy=policy, outer_qids=outer_ids, depth=50, expansion=False, parent_metadata=parent_meta, query_token_lengths=query_token_lengths)
        for index, b_config in enumerate(b_configs):
            model = core.fit_lambdamart(X, y, groups, b_config, feature_names=core.AUGMENTED_FEATURE_NAMES)
            predictions_by_config[index].update(core.rank_model_predictions(model, valid, source_rows, vectors, qrow, support_qids={qid: support for qid in valid}, answers=answers, policy=policy, outer_qids=outer_ids, depth=50, expansion=False, parent_metadata=parent_meta, query_token_lengths=query_token_lengths))
    metrics_by_config = {index: core.prediction_metrics(values, answers, sorted(values)) for index, values in predictions_by_config.items()}
    selected_index = max(metrics_by_config, key=lambda index: (metrics_by_config[index].get('recall@5', -np.inf), metrics_by_config[index].get('multi_gold_recall@5', -np.inf), metrics_by_config[index].get('precision@5', -np.inf), metrics_by_config[index].get('mrr@5', -np.inf), metrics_by_config[index].get('recall@1', -np.inf), -index))
    return {'selected_config': b_configs[selected_index], 'selected_index': selected_index, 'candidates': list(b_configs), 'metrics_by_config': {str(index): value for index, value in metrics_by_config.items()}, 'predictions': predictions_by_config[selected_index], 'selection_scope': 'three training folds only', 'fold0_read': False}

def evaluate_training_context(heldout, policy, residual_config, allow_expansion, b_context):
    train_folds = [fold for fold in core.INNER_FOLDS if fold != heldout]
    alpha, threshold_quantile = residual_config
    threshold_values, anchor_high_confidence_threshold = residual_context_thresholds(train_folds, policy)
    aggregate = {name: {} for name in ('A', 'B', 'C', 'D')}
    for validation in train_folds:
        support = [qid for fold in train_folds if fold != validation for qid in folds[fold]]
        valid = [str(qid) for qid in folds[validation]]
        aggregate['A'].update({qid: anchor_inner[qid] for qid in valid})
        aggregate['C'].update(make_residual_predictions(valid, support, policy, alpha=alpha, threshold_value=threshold_values[threshold_quantile], anchor_high_confidence_threshold=anchor_high_confidence_threshold))
        if allow_expansion:
            aggregate['D'].update(make_expansion_predictions(valid, support, policy))
        else:
            aggregate['D'].update({qid: anchor_inner[qid] for qid in valid})
    aggregate['B'].update(b_context['predictions'])
    metrics = {name: core.prediction_metrics(values, answers, sorted(values)) for name, values in aggregate.items()}
    return aggregate, metrics

winner_predictions, per_fold = {}, {}
all_arm_predictions = {name: {} for name in ('A', 'B', 'C', 'D')}
for heldout in core.INNER_FOLDS:
    valid = [str(qid) for qid in folds[heldout]]
    support = [str(qid) for fold in core.INNER_FOLDS if fold != heldout for qid in folds[fold]]
    chosen = tuple(next(item['policy'] for item in selected_rows if item['heldout'] == heldout))
    b_context = evaluate_b_context(heldout, chosen)
    residual_grid_metrics = {}
    for residual_config in core.RESIDUAL_CONFIG_GRID:
        _train_pred, candidate_metrics = evaluate_training_context(heldout, chosen, residual_config, expansion_gate['eligible'], b_context)
        residual_grid_metrics[tuple(residual_config)] = candidate_metrics
    selected_residual_config = core.select_nested_residual_config(residual_grid_metrics)
    _train_pred, training_metrics = evaluate_training_context(heldout, chosen, selected_residual_config, expansion_gate['eligible'], b_context)
    chosen_arm = core.select_nested_arm(training_metrics)
    train_folds = [fold for fold in core.INNER_FOLDS if fold != heldout]
    threshold_values, anchor_high_confidence_threshold = residual_context_thresholds(train_folds, chosen)
    heldout_arms = {'A': {qid: anchor_inner[qid] for qid in valid}, 'C': make_residual_predictions(valid, support, chosen, alpha=selected_residual_config[0], threshold_value=threshold_values[selected_residual_config[1]], anchor_high_confidence_threshold=anchor_high_confidence_threshold)}
    heldout_arms['D'] = make_expansion_predictions(valid, support, chosen) if expansion_gate['eligible'] else {qid: anchor_inner[qid] for qid in valid}
    X, y, groups, _ids = core.make_augmented_training_rows(support, source_rows, answers, vectors, qrow, support_qids_for_qid={qid: [x for x in support if x != qid] for qid in support}, policy=chosen, outer_qids=outer_ids, depth=50, expansion=False, parent_metadata=parent_meta, query_token_lengths=query_token_lengths)
    model = core.fit_lambdamart(X, y, groups, b_context['selected_config'], feature_names=core.AUGMENTED_FEATURE_NAMES)
    heldout_arms['B'] = core.rank_model_predictions(model, valid, source_rows, vectors, qrow, support_qids={qid: support for qid in valid}, answers=answers, policy=chosen, outer_qids=outer_ids, depth=50, expansion=False, parent_metadata=parent_meta, query_token_lengths=query_token_lengths)
    winner_predictions.update(heldout_arms[chosen_arm])
    for arm_name in all_arm_predictions:
        all_arm_predictions[arm_name].update(heldout_arms[arm_name])
    per_fold[heldout] = {'chosen_arm': chosen_arm, 'policy': list(chosen), 'residual_config': list(selected_residual_config), 'residual_grid_metrics': {str(key): value for key, value in residual_grid_metrics.items()}, 'b_config': b_context, 'training_context_metrics': training_metrics, 'heldout_arm_metrics': {name: core.prediction_metrics(values, answers, valid) for name, values in heldout_arms.items()}, 'selection_scope': 'F1-F4 training-context-only', 'fold0_read': False}

winner_metrics = core.prediction_metrics(winner_predictions, answers, inner_ids)
fold_deltas = {fold: float(per_fold[fold]['heldout_arm_metrics'][per_fold[fold]['chosen_arm']]['recall@5'] - core.prediction_metrics({qid: anchor_inner[qid] for qid in folds[fold]}, answers, folds[fold])['recall@5']) for fold in core.INNER_FOLDS}
deltas, bootstrap = core.paired_delta_bootstrap(anchor_inner, winner_predictions, answers, inner_ids, samples=10000, seed=110)
gate = core.strict_inner_gate(anchor_metrics, winner_metrics, per_fold_delta=fold_deltas, bootstrap=bootstrap)
atomic_json(DRIVE_ROOT / 'cache/prototypes/support_sidecar.json', support_sidecars)
support_by_target = {qid: [str(value) for fold in core.INNER_FOLDS if fold != fold_for[qid] for value in folds[fold]] for qid in inner_ids}
seen_frequency_by_fold = {fold: core.breakdown_seen_frequency(winner_predictions, answers, folds[fold], core.build_support_bank([qid for name in core.INNER_FOLDS if name != fold for qid in folds[name]], vectors, qrow, answers, forbidden_qids=outer_ids, support_fold_names=[name for name in core.INNER_FOLDS if name != fold])) for fold in core.INNER_FOLDS}
arm_metrics = {name: core.prediction_metrics(values, answers, inner_ids) for name, values in all_arm_predictions.items()}
arm_comparison = {name: core.prediction_comparison(anchor_inner, values, answers, inner_ids) for name, values in all_arm_predictions.items() if name != 'A'}
oracle_predictions, oracle_meta = core.choice_oracle_predictions(all_arm_predictions, answers, [qid for qid in inner_ids if answers.get(qid)])
oracle_metrics = core.prediction_metrics(oracle_predictions, answers, [qid for qid in inner_ids if answers.get(qid)])
duplicate_groups = core.exact_normalized_duplicate_groups(np.asarray(vectors[inner_rows], dtype=np.float32), inner_ids)
final_report = {'schema_version': core.SCHEMA, 'stage': 'strict-inner-gate', 'anchor': anchor_metrics, 'arms': arm_metrics, 'winner': winner_metrics, 'per_fold': per_fold, 'per_fold_delta': fold_deltas, 'bootstrap': bootstrap, 'gate': gate, 'arm_comparison_vs_anchor': arm_comparison, 'choice_oracle': {'metrics': oracle_metrics, 'meta': oracle_meta}, 'seen_frequency_breakdown_by_fold': seen_frequency_by_fold, 'nearest_similarity_breakdown': core.nearest_similarity_breakdown(vectors, inner_ids, qrow, support_by_target), 'exact_normalized_query_duplicate_groups': duplicate_groups, 'prototype_coverage': prototype_coverage, 'candidate_expansion': {'gate': expansion_gate, 'by_fold': expansion_rows}, 'support_sidecar_sha256': core.sha256_file(DRIVE_ROOT / 'cache/prototypes/support_sidecar.json'), 'fold0_read': False, 'claim_boundary': 'F1-F4 inner only; not Fold-0/public Recall'}
atomic_json(DRIVE_ROOT / 'results/STRICT_INNER_GATE.json', final_report)
atomic_json(DRIVE_ROOT / 'results/RUN_REPORT.json', {'run_id': RUN_ID, 'status': gate['status'], 'resource_snapshot': snapshot, 'resource_gate': resource_gate, 'gpu_budget': gpu_budget, 'input_manifest_sha256': core.sha256_file(DRIVE_ROOT / 'input/INPUT_MANIFEST.json'), 'strict_inner_gate_sha256': core.sha256_file(DRIVE_ROOT / 'results/STRICT_INNER_GATE.json'), 'fold0_read': False, 'claim_boundary': final_report['claim_boundary']})
atomic_json(DRIVE_ROOT / 'results/DRIVE_UPLOAD_MANIFEST.json', {'run_id': RUN_ID, 'input_manifest': manifest, 'code_files': manifest.get('code_files', {}), 'notebook_expected_path': str(DRIVE_ROOT / 'notebooks/exp110p_semantic_label_prototype_colab.ipynb'), 'verified_input_files': hash_report['files'], 'fold0_read': False})
core.write_jsonl_atomic(DRIVE_ROOT / 'exports/exp110p_selected_predictions.jsonl', ({'qid': qid, 'prediction': winner_predictions[qid], 'fold0_included': False} for qid in sorted(winner_predictions)))
atomic_json(DRIVE_ROOT / 'exports/exp110p_feature_manifest.json', {'schema_version': core.SCHEMA, 'feature_names': list(core.AUGMENTED_FEATURE_NAMES), 'selected_policies': selected_rows, 'fold0_read': False, 'late_interaction_features': False, 'lexical_query_memory': False})
phase_receipt('strict_inner_gate', {'gate': gate, 'bootstrap': bootstrap, 'fold0_read': False}, len(inner_ids), len(inner_ids), run_state=gate['status'])

## Phase 10 boundary — deliberate stop
Fold 0 is not read or scored in this notebook.

In [ ]:
# Terminal cell: deliberate stop boundary. There is no Fold-0 execution cell.
print('STATUS')
print('anchor Recall@5', anchor_metrics.get('recall@5'))
print('winner Recall@5 / delta', winner_metrics.get('recall@5'), gate.get('delta_recall@5'))
print('multi-gold delta', gate.get('multi_gold_delta'))
print('per-fold deltas', fold_deltas)
print('bootstrap CI', bootstrap.get('lower'), bootstrap.get('upper'))
print('seen/unseen breakdown', final_report['seen_frequency_breakdown'])
print('gate result', gate['status'])
print('Fold-0 seen? false')
print('Fold 0 has not been read.')
print('Explicit user authorization is required.')
print('result/log paths on Drive', DRIVE_ROOT / 'results', DRIVE_ROOT / f'results/logs/{RUN_ID}')
print('No public inference or submission was created.')